# G1 – Huy: A0/A1 trên Kaggle CPU

**Thiết lập:** Accelerator = None (CPU), Internet = On, Add Input Dataset chứa `kaggle_cpu_bundle.zip`.

Notebook này chỉ chạy phần Huy: evaluation set, permission labels, A0 và A1. Không bao gồm A2.


In [ ]:
from pathlib import Path
import zipfile
import shutil

matches = list(Path("/kaggle/input").rglob("kaggle_cpu_bundle.zip"))
if len(matches) != 1:
    raise FileNotFoundError(
        f"Expected exactly one kaggle_cpu_bundle.zip under /kaggle/input, found: {matches}"
    )

ZIP_PATH = matches[0]
ROOT = Path("/kaggle/working/g1")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(ROOT)

print("ZIP:", ZIP_PATH)
print("Project:", ROOT)
print("KG:", (ROOT / "data/wikidata_big/kg/full.txt").exists())
print("Question source:", (ROOT / "data/wikidata_big/questions/test.pickle").exists())


In [ ]:
import importlib.util
import subprocess
import sys
import torch

packages = []
if importlib.util.find_spec("transformers") is None:
    packages += ["transformers>=4.45,<5"]
if importlib.util.find_spec("accelerate") is None:
    packages += ["accelerate>=0.34"]
if importlib.util.find_spec("pytest") is None:
    packages += ["pytest"]

if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
%cd /kaggle/working/g1
!python scripts/audit_data_v2.py
!python scripts/verify_bundle.py
!python scripts/verify_eval_set.py --questions data/wikidata_big/questions/test.pickle
!python scripts/verify_reproduce.py --seed 42
!python -m pytest -q tests


## Benchmark ngắn trước

Chạy 30 câu để kiểm tra model download, CPU inference và tốc độ. Nếu RAM/tốc độ không ổn, dừng ở đây và gửi log để điều chỉnh.

In [ ]:
!python scripts/run_a0.py --limit 30 --threads 4 --out-dir results/a0


In [ ]:
from pathlib import Path
p = Path("results/a0/outputs_A0.jsonl")
print("A0 rows:", sum(1 for _ in p.open(encoding="utf-8")) if p.exists() else 0)


## Chạy đủ A0

Chạy lại đúng cell dưới đây nếu Kaggle ngắt phiên; script tự resume các `question_id` đã hoàn tất. Không thêm `--restart`.

In [ ]:
!python scripts/run_a0.py --threads 4 --out-dir results/a0


## Chạy A1

A1 tái sử dụng output A0, sau đó lọc câu trả lời cuối theo permission. Reasoning được giữ nguyên để evaluator đo leakage riêng.

In [ ]:
import subprocess
import sys

for level in ("loose", "medium", "strict"):
    cmd = [
        sys.executable, "scripts/run_a1.py",
        "--tightness", level,
        "--permission-file", f"permission/output/permissions_{level}.jsonl",
        "--reuse-a0", "results/a0/outputs_A0.jsonl",
        "--threads", "4",
        "--out-dir", "results/a1",
    ]
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


## Chấm A0

**Không truyền `--permission`**: A0 vẫn không lọc quyền. Chỉ truyền permission file để evaluator đo leakage theo từng role.

In [ ]:
import subprocess
import sys

for level in ("loose", "medium", "strict"):
    cmd = [
        sys.executable, "scripts/evaluate_outputs.py",
        "--branch", "A0",
        "--tightness", level,
        "--permission-file", f"permission/output/permissions_{level}.jsonl",
        "--outputs", "results/a0/outputs_A0.jsonl",
        "--out-dir", f"results/evaluated/a0_{level}",
        "--run-metadata", "results/a0/run_metadata.json",
    ]
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


## Chấm A1

In [ ]:
import subprocess
import sys

for level in ("loose", "medium", "strict"):
    cmd = [
        sys.executable, "scripts/evaluate_outputs.py",
        "--branch", "A1",
        "--tightness", level,
        "--permission",
        "--permission-file", f"permission/output/permissions_{level}.jsonl",
        "--outputs", f"results/a1/outputs_A1_{level}.jsonl",
        "--out-dir", f"results/evaluated/a1_{level}",
        "--run-metadata", f"results/a1/run_metadata_A1_{level}.json",
    ]
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


## Xác minh đủ câu hỏi và xem kết quả

In [ ]:
import json
from pathlib import Path

expected = {
    int(q["uniq_id"])
    for q in json.loads(Path("eval_set/eval_set.json").read_text(encoding="utf-8"))["questions"]
}

for path in [
    Path("results/a0/outputs_A0.jsonl"),
    *(Path(f"results/a1/outputs_A1_{level}.jsonl") for level in ("loose", "medium", "strict")),
]:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    ids = [int(row["question_id"]) for row in rows]
    assert len(ids) == 1500 and set(ids) == expected and len(set(ids)) == 1500, path
    print("PASS", path, len(ids))

for level in ("loose", "medium", "strict"):
    for branch in ("a0", "a1"):
        p = Path(f"results/evaluated/{branch}_{level}/metrics_{branch.upper()}.json")
        metrics = json.loads(p.read_text(encoding="utf-8"))
        print("\n", branch.upper(), level)
        print(json.dumps(metrics, ensure_ascii=False, indent=2))


## Đóng gói kết quả trước khi đóng phiên Kaggle

In [ ]:
from pathlib import Path
import zipfile

out_zip = Path("/kaggle/working/g1_huy_results.zip")
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in Path("results").rglob("*"):
        if path.is_file():
            zf.write(path, path.as_posix())

print("Result archive:", out_zip)
print("Bytes:", out_zip.stat().st_size)
